1. 内容生成器（LLM）
2. 结构控制器（JSON schema）
3. 事实约束器（grounding）
4. 安全与质量校验器（validator）
5. 呈现与交互层（UI / human-in-the-loop）

                ┌──────────────────────┐
                │   1. 内容生成器      │
                │   (LLM 写邮件)       │
                └────────┬─────────────┘
                         ↓
                ┌──────────────────────┐
                │   2. 结构控制器      │
                │   (JSON schema)      │
                └────────┬─────────────┘
                         ↓
                ┌──────────────────────┐
                │   3. 事实约束器      │
                │   (RAG grounding)    │
                └────────┬─────────────┘
                         ↓
                ┌──────────────────────┐
                │   4. 校验器          │
                │   (rules + safety)   │
                └────────┬─────────────┘
                         ↓
                ┌──────────────────────┐
                │   5. 呈现层          │
                │   (human review UI)  │
                └──────────────────────┘

In [3]:
# 读取openai key
from dotenv import load_dotenv
import os
from pathlib import Path

# 当前路径
BASE_DIR = Path.cwd()

# 自动找到 project root
if (BASE_DIR / ".env").exists():
    env_path = BASE_DIR / ".env"
else:
    env_path = BASE_DIR.parent / ".env"

# 加载
load_dotenv(env_path)

# 读取
api_key = os.getenv("OPENAI_API_KEY")

print("API KEY:", api_key[:10], "...")

API KEY: sk-proj-QH ...


In [9]:
import json
from openai import OpenAI

client = OpenAI()

In [10]:
agent_input = {
    "customer_name": "Dr. Smith",
    "subject": "Inquiry about CD19 CAR-T products",
    "customer_email": "Hello, I am interested in your CD19 CAR-T products. Could you share pricing and lead time?",
    "detected_intent": "product_inquiry",
    "retrieved_context": [
        {
            "source": "CAR-T Products Brochure.txt",
            "content": "Catalog No: PM-CAR1067 | Target Antigen: CD19 | Costimulatory Domain: 4-1BB | Price: $1259"
        },
        {
            "source": "General Service FAQ.txt",
            "content": "Lead time should be confirmed with the sales or production team depending on product type and availability."
        }
    ],
    "company_name": "ProMab Biotechnologies",
    "sender_name": "Suna",
    "sender_title": "Customer Support",
    "tone": "professional"
}

In [11]:
import json
from typing import Any, Dict, List, Optional
from openai import OpenAI

client = OpenAI()

SYSTEM_PROMPT = """
You are a professional biotech customer support email drafting assistant.

Your job is to draft high-quality customer email replies for a biotech company.
You must sound professional, polite, clear, concise, and helpful.

Core rules:
1. Write in formal email style, not chat style.
2. Base the reply only on the customer email and the retrieved context provided.
3. Do not invent product details, pricing, lead time, technical claims, policies, or availability.
4. If a requested detail is not clearly supported by the retrieved context, state that it needs confirmation.
5. If appropriate, suggest a practical next step, such as confirming lead time, sharing additional product options, or requesting more details.
6. Keep the draft customer-ready and natural, as if written by a trained biotech support specialist.
7. Do not mention internal system logic, retrieval process, or source file names in the final email draft.
8. If the customer asks multiple questions, address each one clearly.
9. If information is insufficient, respond helpfully without overclaiming.
10. Return valid JSON only.

Required JSON fields:
- detected_intent: string
- used_sources: list of strings
- answer_points: list of strings
- uncertainties: list of strings
- missing_information: list of strings
- draft_email_subject: string
- draft_email_body: string
- confidence: one of ["high", "medium", "low"]
"""

In [12]:
def build_user_prompt(agent_input: Dict[str, Any]) -> str:
    customer_name = agent_input.get("customer_name", "")
    subject = agent_input.get("subject", "")
    customer_email = agent_input.get("customer_email", "")
    detected_intent = agent_input.get("detected_intent", "unknown")
    retrieved_context = agent_input.get("retrieved_context", [])
    company_name = agent_input.get("company_name", "")
    sender_name = agent_input.get("sender_name", "")
    sender_title = agent_input.get("sender_title", "")
    tone = agent_input.get("tone", "professional")

    if retrieved_context:
        context_blocks = []
        for i, item in enumerate(retrieved_context, start=1):
            source = item.get("source", f"Source {i}")
            content = item.get("content", "")
            context_blocks.append(
                f"[Retrieved Context {i}]\nSource: {source}\nContent: {content}"
            )
        context_text = "\n\n".join(context_blocks)
    else:
        context_text = "No retrieved context provided."

    return f"""
Company name:
{company_name}

Sender name:
{sender_name}

Sender title:
{sender_title}

Preferred tone:
{tone}

Customer name:
{customer_name}

Original email subject:
{subject}

Detected intent:
{detected_intent}

Customer email:
{customer_email}

Retrieved context:
{context_text}

Please produce a professional reply draft in valid JSON.
The email should be customer-ready, concise but complete, and suitable for a biotech support workflow.
"""

In [13]:
def validate_reply_output(result: Dict[str, Any]) -> Dict[str, Any]:
    expected_defaults = {
        "detected_intent": "unknown",
        "used_sources": [],
        "answer_points": [],
        "uncertainties": [],
        "missing_information": [],
        "draft_email_subject": "",
        "draft_email_body": "",
        "confidence": "low",
    }

    for key, default_value in expected_defaults.items():
        if key not in result:
            result[key] = default_value

    # Type normalization
    list_fields = ["used_sources", "answer_points", "uncertainties", "missing_information"]
    for field in list_fields:
        if not isinstance(result[field], list):
            result[field] = [str(result[field])]

    str_fields = ["detected_intent", "draft_email_subject", "draft_email_body", "confidence"]
    for field in str_fields:
        if not isinstance(result[field], str):
            result[field] = str(result[field])

    if result["confidence"] not in {"high", "medium", "low"}:
        result["confidence"] = "low"

    return result

In [14]:
def draft_reply(
    agent_input: Dict[str, Any],
    model: str = "gpt-4.1-mini"
) -> Dict[str, Any]:
    """
    Generate a professional customer email draft based on user email and retrieved context.

    Parameters
    ----------
    agent_input : dict
        Structured input including customer email, intent, retrieved context, and sender info.
    model : str
        OpenAI model name.

    Returns
    -------
    dict
        Structured output including answer points, uncertainties, and final email draft.
    """
    user_prompt = build_user_prompt(agent_input)

    response = client.responses.create(
        model=model,
        input=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": user_prompt},
        ],
    )

    raw_text = response.output_text.strip()

    try:
        result = json.loads(raw_text)
    except json.JSONDecodeError:
        result = {
            "detected_intent": agent_input.get("detected_intent", "unknown"),
            "used_sources": [],
            "answer_points": [],
            "uncertainties": ["Model did not return valid JSON."],
            "missing_information": [],
            "draft_email_subject": agent_input.get("subject", "Re: Customer Inquiry"),
            "draft_email_body": raw_text,
            "confidence": "low",
        }

    return validate_reply_output(result)

In [15]:
def render_email(reply_output: Dict[str, Any]) -> str:
    subject = reply_output.get("draft_email_subject", "").strip()
    body = reply_output.get("draft_email_body", "").strip()
    return f"Subject: {subject}\n\n{body}"

In [16]:
agent_input = {
    "customer_name": "Dr. Smith",
    "subject": "Inquiry about CD19 CAR-T products",
    "customer_email": (
        "Hello, I am interested in your CD19 CAR-T products. "
        "Could you please share pricing and lead time?"
    ),
    "detected_intent": "product_inquiry",
    "retrieved_context": [
        {
            "source": "CAR-T Products Brochure.txt",
            "content": (
                "Catalog No: PM-CAR1067 | Target Antigen: CD19 | "
                "Costimulatory Domain: 4-1BB | Price: $1259"
            )
        },
        {
            "source": "General Service FAQ.txt",
            "content": (
                "Lead time should be confirmed with the sales or production team "
                "depending on product type and availability."
            )
        }
    ],
    "company_name": "ProMab Biotechnologies",
    "sender_name": "Suna",
    "sender_title": "Customer Support",
    "tone": "professional"
}

result = draft_reply(agent_input)
print(json.dumps(result, indent=2, ensure_ascii=False))
print()
print(render_email(result))

{
  "detected_intent": "product_inquiry",
  "used_sources": [
    "CAR-T Products Brochure.txt",
    "General Service FAQ.txt"
  ],
  "answer_points": [
    "The price for the CD19 CAR-T product (Catalog No: PM-CAR1067) is $1259.",
    "Lead time for the product needs to be confirmed with the sales or production team due to variability in product type and availability."
  ],
  "uncertainties": [
    "Exact lead time for the CD19 CAR-T product."
  ],
  "missing_information": [
    "Specific requirements or quantities that may affect lead time.",
    "Whether the customer would like assistance connecting with the sales or production team."
  ],
  "draft_email_subject": "Information on CD19 CAR-T Product Pricing and Lead Time",
  "draft_email_body": "Dear Dr. Smith,\n\nThank you for your inquiry regarding our CD19 CAR-T products. The price for the product with Catalog No: PM-CAR1067, which targets the CD19 antigen and contains a 4-1BB costimulatory domain, is $1259.\n\nRegarding lead time